In [1]:
import streamlit as st
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import astropy.units as u
import astropy.constants as const
from astropy.io import fits
import scipy 
import urllib.request
import os 
from scipy import optimize
import photutils.detection as detect

In [2]:
NGC2660_555_fits = "hst_10634_03_acs_wfc_f555w_j9dm03_drc.fits"
NGC2660_814_fits = "hst_10634_03_acs_wfc_f814w_j9dm03_drc.fits"
NGC2660_fits = "hst_10634_03_acs_wfc_total_j9dm03.fits"
# These are the fits files that we used to get the positions and intensities of the stars (from the pixel values and positions).
NGC2660_fits_url = "https://hla.stsci.edu/cgi-bin/getdata.cgi?config=ops&dataset=hst_10634_03_acs_wfc_total_j9dm03"
NGC2660_555_fits_url = "https://hla.stsci.edu/cgi-bin/getdata.cgi?config=ops&download=1&dataset=hst_10634_03_acs_wfc_total_j9dm03&filename=hst_10634_03_acs_wfc_f555w_j9dm03_drc.fits"
NGC2660_814_fits_url = "https://hla.stsci.edu/cgi-bin/getdata.cgi?config=ops&download=1&dataset=hst_10634_03_acs_wfc_total_j9dm03&filename=hst_10634_03_acs_wfc_f814w_j9dm03_drc.fits"
# These are the urls we used to store the fits files, as Github does not allow us to upload them (they are too large).

if not os.path.exists(NGC2660_814_fits) & os.path.exists(NGC2660_555_fits):
        try:
            urllib.request.urlretrieve(NGC2660_fits_url, NGC2660_fits)
            print(f"Downloaded {NGC2660_fits}")
            urllib.request.urlretrieve(NGC2660_814_fits_url, NGC2660_814_fits)
            print(f"Downloaded {NGC2660_814_fits}")
            urllib.request.urlretrieve(NGC2660_555_fits_url, NGC2660_555_fits)
            print(f"Downloaded {NGC2660_555_fits}")
        except Exception as e:
            print("Error Downloading File")
            NGC2660_555_fits = None
            NGC2660_814_fits = None
#This is the code we run to download the fits files from the urls. If you do not have them downloaded, this automatically downloads them for you. If you do have them downloaded, it will skip this step.

In [3]:
HDUList = fits.open(NGC2660_fits)

image_total = HDUList[1].data
#This is the total image of the cluster.

plt.figure()
plt.imshow(image_total, cmap='gray', origin = 'lower', norm=LogNorm())
plt.colorbar(norm=LogNorm())
plt.show()
#Plotting the total image of the cluster (from HST).

plt.savefig('NGC2660_total.png')
#Saving the image as a png file to be used in the Streamlit!

HDUList.close()

C:\Users\awalk\AppData\Local\Temp\ipykernel_21368\817612454.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
bands = [NGC2660_555_fits, NGC2660_814_fits]
#These are the bands needed for the color-magnitude diagram (555nm and 814 nm).
image_data = []
for b in bands:
    with fits.open(b) as HDUList:
        image_data.append(HDUList[1].data) 
        #This reads the data from the SCI section of the fits file...which is the image data.
        HDUList.close()
        #This closes the fits file, which is important to do in order to avoid too much resource usage.
image_555, image_814 = image_data

In [5]:
clipped_data_555 = image_555.copy()
for i in range(5):
    mean = np.nanmean(clipped_data_555)
    #This takes the mean of the clipped data while ignoring NaN values (not a number).
    std = np.nanstd(clipped_data_555)
    #This is the standard deviation of the clipped data (which also ignores NaN values).
    mask = np.abs(clipped_data_555 - mean) <= 3 * std
    #This masks the background
    clipped_data = clipped_data_555[mask]
    background_mean_555 = np.mean(clipped_data)
#This is finding the background for the background subtraction for the 555nm band, as the background would have caused issues in finding the stars.



clipped_data_814 = image_814.copy()
for i in range(5):
    mean = np.nanmean(clipped_data_814)
    std = np.nanstd(clipped_data_814)
    mask = np.abs(clipped_data_814 - mean) <= 3 * std
    clipped_data = clipped_data_814[mask]
    background_mean_814 = np.mean(clipped_data)
#This is the exact same thing, but for the 814nm band.

In [6]:
subtracted_image_555 = image_555 - background_mean_555
#This is the background subtracted image for the 555nm band, which is the original 555nm image and we subtract what we found in the last code cell from it.

plt.figure()
plt.imshow(subtracted_image_555, cmap='grey', origin = 'lower', norm=LogNorm())
plt.colorbar(norm=LogNorm())
plt.show()
#Using matplotlib, we can plot the background subtracted image for the 555nm band, with these lines of code. The imshow is showing the images with a colormap of our choosing. It also adds in the scale that we chose to use (which was logarithmic). The colorbar shows the intensity of the pixels, and it is also on a log-based scale. We chose logarithmic because it helps show the dimmer stars.

fig, axes = plt.subplots()
axes.hist(image_555.flatten(), bins=50, color='blue', range=(-.1, .75), alpha=.6)
axes.hist(subtracted_image_555.flatten(), bins=50, color='red', range=(-.1, .75), alpha=.6)
#This makes two different histograms, one for the original image (blue) and one for the background subtracted image (red). We are showing this to illustrate the difference in data for the subtracted and non-subtracted images. It also helps us see if the background subtraction worked the way we wanted it to.

C:\Users\awalk\AppData\Local\Temp\ipykernel_21368\550122910.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


(array([6.220000e+02, 1.470100e+04, 6.970070e+05, 6.546433e+06,
        6.149396e+06, 1.360588e+06, 4.191260e+05, 2.209370e+05,
        1.453760e+05, 1.051600e+05, 8.035400e+04, 6.481100e+04,
        5.367200e+04, 4.488800e+04, 3.758500e+04, 3.253500e+04,
        2.810000e+04, 2.442800e+04, 2.146900e+04, 1.908600e+04,
        1.674400e+04, 1.511500e+04, 1.332900e+04, 1.196300e+04,
        1.061500e+04, 9.828000e+03, 9.078000e+03, 8.166000e+03,
        7.558000e+03, 7.030000e+03, 6.450000e+03, 5.924000e+03,
        5.632000e+03, 5.145000e+03, 4.856000e+03, 4.717000e+03,
        4.269000e+03, 3.979000e+03, 3.837000e+03, 3.568000e+03,
        3.366000e+03, 3.216000e+03, 3.042000e+03, 3.020000e+03,
        2.826000e+03, 2.620000e+03, 2.511000e+03, 2.406000e+03,
        2.289000e+03, 2.338000e+03]),
 array([-0.1       , -0.083     , -0.066     , -0.049     , -0.032     ,
        -0.015     ,  0.002     ,  0.019     ,  0.036     ,  0.053     ,
         0.07      ,  0.087     ,  0.104     ,  

In [7]:
subtracted_image_555 = image_555 - background_mean_555

plt.figure()
plt.imshow(subtracted_image_555, cmap='grey', origin = 'lower', norm=LogNorm())
plt.colorbar(norm=LogNorm())
plt.show()

fig, axes = plt.subplots()
axes.hist(image_555.flatten(), bins=50, color='blue', range=(-.1, .75), alpha=.6)
axes.hist(subtracted_image_555.flatten(), bins=50, color='red', range=(-.1, .75), alpha=.6)
#This is the same thing as the last code cell but for the 814nm band.

C:\Users\awalk\AppData\Local\Temp\ipykernel_21368\1037366954.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


(array([6.220000e+02, 1.470100e+04, 6.970070e+05, 6.546433e+06,
        6.149396e+06, 1.360588e+06, 4.191260e+05, 2.209370e+05,
        1.453760e+05, 1.051600e+05, 8.035400e+04, 6.481100e+04,
        5.367200e+04, 4.488800e+04, 3.758500e+04, 3.253500e+04,
        2.810000e+04, 2.442800e+04, 2.146900e+04, 1.908600e+04,
        1.674400e+04, 1.511500e+04, 1.332900e+04, 1.196300e+04,
        1.061500e+04, 9.828000e+03, 9.078000e+03, 8.166000e+03,
        7.558000e+03, 7.030000e+03, 6.450000e+03, 5.924000e+03,
        5.632000e+03, 5.145000e+03, 4.856000e+03, 4.717000e+03,
        4.269000e+03, 3.979000e+03, 3.837000e+03, 3.568000e+03,
        3.366000e+03, 3.216000e+03, 3.042000e+03, 3.020000e+03,
        2.826000e+03, 2.620000e+03, 2.511000e+03, 2.406000e+03,
        2.289000e+03, 2.338000e+03]),
 array([-0.1       , -0.083     , -0.066     , -0.049     , -0.032     ,
        -0.015     ,  0.002     ,  0.019     ,  0.036     ,  0.053     ,
         0.07      ,  0.087     ,  0.104     ,  

In [8]:
sigma = 3
#This is the sigma value for the gaussian PSF, which is the uncertainty of the star's intensity.

fwhm = 2.355 * sigma
#This is the full width at half maximum value, which is the width of the signal peak of a star. 

subtracted_image_555 = image_555 - background_mean_555
subtracted_image_814 = image_814 - background_mean_814
#The subtracted images for the bands, already done in a different code cell.
print(background_mean_555)
detection_threshold =  5 * background_mean_555
print(detection_threshold)
filter = int(np.ceil(fwhm))
#This is the detection threshold, which is 5 times the background mean. This is used to detect the stars. We are looking at the 555 nm band at the moment.


detections =  detect.find_peaks(data=subtracted_image_555, threshold=detection_threshold, mask=mask*1000, min_separation=50 , n_peaks=200)
print(detections)
detections_x = detections['x_peak']
detections_y = detections['y_peak']
#This is use to detect the peaks of the stars in the image, which is used to find the intensities of the stars.

plt.figure()
plt.imshow(subtracted_image_555, cmap='grey', origin = 'lower', norm=LogNorm(vmin = .0001, vmax=300))
plt.scatter(detections_x, detections_y, color = 'purple', alpha =.6, marker='o', )
plt.colorbar(norm=LogNorm(vmin = .0001, vmax=300))
plt.show()

0.09751291
0.48756453
 id x_peak y_peak peak_value
--- ------ ------ ----------
  1   3231   2008  242.41061
  2   4107    968  235.84583
  3   5187   3702  233.49498
  4   3634    934  233.25421
  5   3190   2210  232.13773
  6   4101   1408  227.68533
  7   3747   1066  226.90038
  8   2777   1778   226.2516
  9   3616   1573  225.79086
 10   3350   1728  224.94804
...    ...    ...        ...
191   2298   1434  196.79942
192   2868   4146  196.55034
193   1889   3342  196.52748
194   1813   2877  196.08607
195   3262   3491  195.78769
196   3001   1816  195.69519
197   2903   3772  195.56503
198   2368   1188  194.88678
199   2897   2941  194.53185
200   2640   4389  194.50577
Length = 200 rows


C:\Users\awalk\AppData\Local\Temp\ipykernel_21368\127677636.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
class Star:
    """One star is the coordinates of the center of a star on the image"""
    
    def __init__(self, row, col, name=""):
        self.row = row
        self.col = col
        self.name = name
    #The definition of the star class, where you can then call upon the row, column and name.

    def get_row_col(self):
        return (self.row, self.col)
    #This is the function to get the row and column of the star.

    def value_at(self, image):
        return image[self.row, self.col]
    #This gets the value of the star at a given row and column and then return it as an image.
        

    def cutout(self, image, half_size=8):
        nrows, ncols = image.shape
        r_start = self.row - half_size
        r_end = self.row + half_size + 1
        c_start = self.col - half_size + 1
        c_end = self.col + half_size + 1
        if r_start < 0:
            r_start = 0
        if c_start < 0: 
            c_start = 0
        if r_end > nrows:
            r_end = nrows
        if c_end > ncols:
            c_end = ncols
        return image[r_start:r_end, c_start:c_end]
        #This makes the cutout of the star, which is the star's image. This also makes sure that the cutout does not go past the border of the image.
#This is the class for the stars, which is used to get the positions of the stars. This then is used in the next cell to find the PSF fitting.


In [10]:
stars = [
    Star(1311, 1231, "1"),
    Star(544, 3297, "2"),
    Star(2910, 4128, "3"),
    Star(4462, 1557, "4"),
    Star(1657, 2605, "5"),
    Star(4064, 1437, "6"),
    Star(2160, 690, "7"),
    Star(4531, 1714, "8"),
    Star(2010, 3230, "9"),
    Star(2710, 1710, "10"),
    Star(4531, 1713, "11"),
    Star(4168, 2647, "12"),
    Star(3565, 3130, "13"),
    Star(3203, 1623, "14"),
    Star(2228, 2120, "15"),
    Star(2888, 2795, "16"),
    Star(2015, 2177, "18"),
    Star(3056, 1063, "19"),
    Star(4116, 3859, "20"),
    Star(3333, 866, "21"),
    Star(2340, 1630, "22"),
    Star(1880, 2087, "23"),
    Star(1900, 1738, "24"),
    Star(2003, 1967, "25"),
    Star(1953, 1999, "26"),
    Star(1988, 2106, "27"),
    Star(2016, 2176, "28"),
    Star(2193, 2216, "29"),
    Star(1433, 2298, "30")
    ]

#Manually detected stars, which we put into a list for the PSF fitting.

In [11]:
flux = []

def psf_fit(coords, row, col, sigma=3, amplitude_func=10,  offset=0):
    x, y = coords
    

    psf = amplitude_func * np.exp(-((y-row)**2 + (x-col)**2)/(2*sigma**2))
    fwhm = 2.355*sigma
    return psf.ravel()
    #Making 2D circular gaussian PSF. The ravel is used to make the PSF a 1D array, which is needed for the image_2660 function.

def image_2660(selected_image):
    """Performs two curve fits using data from two fits files.
        Selected image is the data we use with the subtracted background.
        Image_2660 takes the cutouts of the stars that we selected via the star class.
        This then attempts to remove their diffraction spikes to better fit the stars."""
    for s in stars:
        cutout = s.cutout(image=selected_image)
        nonan_cutout = np.nan_to_num(cutout)
        y_size, x_size = nonan_cutout.shape
        yy, xx = np.meshgrid(np.arange(y_size), np.arange(x_size), indexing='ij') 
        coordinates = (yy, xx)
        amplitude = nonan_cutout.ravel()
        amplitude = np.nan_to_num(amplitude, nan=np.nanmedian(amplitude))
        peak = np.max(amplitude)
        bg = np.median(amplitude)
        #This is for finding the peak intensities of the cutouts of chosen stars.
    

        lower_bounds = [0, 0, 0.5, 0, -np.inf]
        upper_bounds = [y_size, x_size, 10.0, np.inf, np.inf]
        #These are the bounds for the curve fit itself.
        
        p0 = [
        y_size / 2,
        x_size / 2,
        1.0,
        peak - bg,
        bg

        ]
        popt, pcov = optimize.curve_fit(
            psf_fit,
            coordinates,
            amplitude,
            p0=p0,
            bounds=(lower_bounds, upper_bounds),
            maxfev=20000
            )         
        actual_data = amplitude.reshape(y_size, x_size)
        #This is the original star image, which we then use to make a curve fit of it.
        first_fit_model = psf_fit(coordinates, *popt).reshape(y_size, x_size)
        mask_data = (actual_data - first_fit_model) > 60.
        actual_data[mask_data] = np.nan
        nonan_masked = np.nan_to_num(actual_data, nan=background_mean_555)
        masked_popt, mask_pcov = optimize.curve_fit(
            psf_fit,
            coordinates,
            nonan_masked.ravel(),
            p0=p0,
            bounds = (lower_bounds, upper_bounds),
            maxfev=500000,
            )
        #This is the work that is done in order to make the curve fit in the first place.

        second_fit_model = psf_fit(coordinates, *masked_popt).reshape(y_size, x_size)
        actual_data = amplitude.reshape(y_size, x_size)
        fig, ax = plt.subplots(1, 3, figsize=(15, 5))
        peak_final = np.max(nonan_masked) 
        flux.append(2 * np.pi * peak_final * 2**2)
        im0 = ax[0].imshow(actual_data, origin='lower')
        ax[0].set_title('Actual Star (Data)')
        fig.colorbar(im0, ax=ax[0])
        averaged_fit = (first_fit_model+second_fit_model)/2
        im1 = ax[1].imshow(averaged_fit, origin='lower')
        ax[1].set_title('Best Fit')
        fig.colorbar(im1, ax=ax[1])
        
        im2 = ax[2].imshow(actual_data - averaged_fit, origin='lower')
        ax[2].set_title('Residuals (Data - Model)')
        fig.colorbar(im2, ax=ax[2]) 

image_2660(subtracted_image_555)
image_2660(subtracted_image_814)
print(f"flux {flux}")

C:\Users\awalk\AppData\Local\Temp\ipykernel_21368\830509540.py:68: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, ax = plt.subplots(1, 3, figsize=(15, 5))


flux [np.float32(5266.194), np.float32(80.236084), np.float32(246.13202), np.float32(226.5246), np.float32(688.84985), np.float32(5062.9644), np.float32(5565.3105), np.float32(895.65576), np.float32(6092.4434), np.float32(5387.8237), np.float32(895.65576), np.float32(4872.5635), np.float32(4952.0356), np.float32(5169.1304), np.float32(5374.3086), np.float32(5366.43), np.float32(5370.638), np.float32(5394.1333), np.float32(5017.021), np.float32(5138.2866), np.float32(5615.385), np.float32(5501.2393), np.float32(5426.742), np.float32(5426.119), np.float32(5024.8594), np.float32(5402.5225), np.float32(5370.638), np.float32(5593.6367), np.float32(4946.109), np.float32(5655.5566), np.float32(457.66074), np.float32(900.6031), np.float32(915.0482), np.float32(2460.4385), np.float32(5149.4546), np.float32(5656.491), np.float32(2834.2292), np.float32(6058.101), np.float32(5603.888), np.float32(2834.2292), np.float32(5208.326), np.float32(5260.482), np.float32(5194.0996), np.float32(5605.9565), 

In [12]:
flux_split = len(flux)//2 
intensity_555 = np.asarray(flux[:flux_split])
intensity_814 = np.asarray(flux[flux_split:])
print(f"flux_555 {intensity_555}")
print(f"flux_814 {intensity_814}")
flux_subtraction = intensity_555 - intensity_814
print(f"flux_subtraction {flux_subtraction}")
#This is the flux of both the bands, which we then subtract from each other to get the flux subtraction. The flux subtraction is then used for the color-magnitude diagram as the color.
plt.figure()
flux_plot = plt.scatter(flux_subtraction, intensity_814)
plt.xlim(-400, 200)
plt.ylim(5000, 6150)
plt.xlabel("Color")
plt.ylabel("Magnitude")
plt.title("NGC 2660 Color-Magnitude Diagram")
plt.show()

plt.savefig('NGC2660_CMD.png')


#This is the color-magnitude diagram, which is the plot of the flux subtraction vs the 814nm band flux. This is the end-goal of the project.

flux_555 [5266.194      80.236084  246.13202   226.5246    688.84985  5062.9644
 5565.3105    895.65576  6092.4434   5387.8237    895.65576  4872.5635
 4952.0356   5169.1304   5374.3086   5366.43     5370.638    5394.1333
 5017.021    5138.2866   5615.385    5501.2393   5426.742    5426.119
 5024.8594   5402.5225   5370.638    5593.6367   4946.109   ]
flux_814 [5655.5566   457.66074  900.6031   915.0482  2460.4385  5149.4546
 5656.491   2834.2292  6058.101   5603.888   2834.2292  5208.326
 5260.482   5194.0996  5605.9565  5347.416   5400.916   5362.0947
 5186.305   5162.179   5885.2944  5700.301   5700.6494  5585.651
 5332.455   5416.779   5400.916   5658.328   5506.515  ]
flux_subtraction [ -389.3628    -377.42465   -654.47107   -688.5236   -1771.5886
   -86.490234   -91.180664 -1938.5735      34.342285  -216.06445
 -1938.5735    -335.7627    -308.4463     -24.969238  -231.64795
    19.01416    -30.277832    32.038574  -169.28418    -23.892578
  -269.90967   -199.06152   -273.90723   

C:\Users\awalk\AppData\Local\Temp\ipykernel_21368\3302723791.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:
#Streamlit code:
st.title("NGC 2660 Color-Magnitude Diagram")
st.write("The image of the total cluster is shown below:")
st.image('NGC2660_total.png')
st.write("This is the Color-Magnitude Diagram of NGC 2660, which is an open cluster of stars.")

2026-04-27 23:30:45.574 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-27 23:30:45.654 
  command:

    streamlit run c:\Users\awalk\anaconda3\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-04-27 23:30:45.655 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-27 23:30:45.656 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-27 23:30:45.657 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-27 23:30:45.658 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-27 23:30:45.659 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-27 23:30:45.659 Thread 'MainThread': mi